In [ ]:
"""
Step 3b of 5 — historical accuracy and the induced operator (IOWAGDP).

Self-contained: upload the two inputs when prompted, outputs download at the end.

UPLOAD   aggregates.csv       iso, year, IMF, OECD, EC, owa_0.8   (02_aggregate)
         forecast_errors.csv  iso, target, IMF, OECD, EC, actual  (03a_build_errors)
WRITES   mae.csv              per-country MAE, induced order, separation
         induced.csv          IOWAGDP vs OWAGDP per cell, both inducing variables

u = 1/MAE enters only through the ORDER, so this is equivalent to ranking the
institutions by ascending MAE; the reciprocal carries no magnitude. Ties in u
are broken by descending forecast, which is the convention that makes the bound
in Proposition 3 tight.
"""
import os
import numpy as np
import pandas as pd

INST = ["IMF", "OECD", "EC"]
A_HI = 0.8
MIN_ROUNDS = 3
COVID_TARGET = 2021
# publication order of the current vintages, newest first: OECD Jun 2026,
# EC May 2026, IMF Apr 2026. Identical in every country by construction.
RECENCY = ["OECD", "EC", "IMF"]

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def grab(cols, label):
    def scan():
        for f in sorted(os.listdir(".")):
            if f.lower().endswith(".csv"):
                try:
                    if cols.issubset(pd.read_csv(f, nrows=1).columns):
                        return f
                except Exception:
                    pass
        return None
    f = scan()
    if f is None and IN_COLAB:
        print(f"Upload {label}")
        files.upload()
        f = scan()
    if f is None:
        raise FileNotFoundError(
            f"no csv here with columns {sorted(cols)} ({label}).\n"
            f"Present: {sorted(os.listdir('.'))}")
    print(f"{label:<20}{f}")
    return pd.read_csv(f)


agg = grab({"iso", "year", "owa_0.8", *INST}, "aggregates.csv")
err = grab({"iso", "target", "actual", *INST}, "forecast_errors.csv")
print()


def me_weights(alpha, n=3):
    if abs(alpha - 0.5) < 1e-12:
        return np.full(n, 1.0 / n)

    def orness(h):
        w = h ** np.arange(n)
        w = w / w.sum()
        return ((n - 1 - np.arange(n)) * w).sum() / (n - 1), w

    lo, hi = 1e-9, 1e9
    for _ in range(200):
        h = np.sqrt(lo * hi)
        lo, hi = (h, hi) if orness(h)[0] > alpha else (lo, h)
    return orness(np.sqrt(lo * hi))[1]


W = me_weights(A_HI)

# ------------------------------------------------------------------- MAE ----
for k in INST:
    err["s_" + k] = err[k] - err["actual"]          # signed error = fcst - actual

mae = err.groupby("iso")[["s_" + k for k in INST]].agg(lambda s: s.abs().mean())
mae.columns = INST
rounds = err.groupby("iso")[["s_" + k for k in INST]].count()
rounds.columns = INST

dropped = sorted(rounds.index[(rounds < MIN_ROUNDS).any(axis=1)])
mae = mae[(rounds >= MIN_ROUNDS).all(axis=1)]
if dropped:
    print(f"dropped for fewer than {MIN_ROUNDS} usable rounds: {', '.join(dropped)}\n")

mae["best"] = mae[INST].idxmin(axis=1)
mae["order_acc"] = ["-".join(sorted(INST, key=lambda k: r[k]))
                    for _, r in mae.iterrows()]
# the SMALLEST gap between ADJACENT ranked MAEs -- min over both adjacent
# pairs, not just the first. Ireland separates ranks 1 and 2 comfortably but
# ranks 2 and 3 by 0.004 points, and it is that second gap that matters.
vals = np.sort(mae[INST].values, axis=1)          # ascending
adj = np.diff(vals, axis=1)                       # [g2-g1, g3-g2]
j = adj.argmin(axis=1)                            # which adjacent pair is closest
rows = np.arange(len(mae))
mae["min_gap"] = adj[rows, j]
pair_mean = (vals[rows, j] + vals[rows, j + 1]) / 2
mae["min_gap_pct"] = mae["min_gap"] / pair_mean * 100    # % of THAT pair's mean
mae.round(4).to_csv("mae.csv")

# ------------------------------------------------- induced aggregation ------
out = []
for _, r in agg.iterrows():
    if r.iso not in mae.index:
        continue
    g = {k: r[k] for k in INST}
    m = mae.loc[r.iso]
    by_mag = sorted(INST, key=lambda k: -g[k])
    by_acc = sorted(INST, key=lambda k: (m[k], -g[k]))    # tie -> larger value
    swap = by_acc[:1] + [by_acc[2], by_acc[1]]            # exchange ranks 2 and 3
    out.append({
        "iso": r.iso, "year": r.year,
        "owa": float(np.array([g[k] for k in by_mag]) @ W),
        "iowa_acc": float(np.array([g[k] for k in by_acc]) @ W),
        "iowa_rec": float(np.array([g[k] for k in RECENCY]) @ W),
        "iowa_acc_swap23": float(np.array([g[k] for k in swap]) @ W),
        "order_acc": "-".join(by_acc),
        "min_gap": m.min_gap,
        "min_gap_pct": m.min_gap_pct,
    })
d = pd.DataFrame(out)
d["diff_acc"] = d.iowa_acc - d.owa
d["diff_rec"] = d.iowa_rec - d.owa
d["swap_effect"] = (d.iowa_acc_swap23 - d.iowa_acc).abs()
d.round(4).to_csv("induced.csv", index=False)

# ---------------------------------------------------------------- report ----
print(f"{len(mae)} economies with a complete record -> {len(d)} cells\n")
print("Section V-C")
for k in INST:
    print(f"  {k:<4} most accurate in {(mae.best==k).sum():3d} | "
          f"panel MAE {mae[k].mean():.3f}")

nz = d.diff_acc.abs() > 1e-9
print(f"\n  accuracy ordering changes the aggregate in {nz.sum()} of {len(d)} cells")
print(f"  mean |difference| {d.diff_acc.abs().mean():.3f} pp over all {len(d)} cells"
      f"  ({d.loc[nz,'diff_acc'].abs().mean():.3f} over the {nz.sum()} that change)")
print(f"  Proposition 3 violations (difference > 0): {(d.diff_acc > 1e-9).sum()}")

nzr = d.diff_rec.abs() > 1e-9
print(f"\n  recency ordering changes {nzr.sum()} of {len(d)} cells, mean "
      f"|difference| {d.diff_rec.abs().mean():.3f} pp over all cells "
      f"({d.loc[nzr,'diff_rec'].abs().mean():.3f} over those that change)")
print("  (larger displacement is not evidence of a better inducing variable:")
print("   the recency order is identical in every country)")

print(f"\n  economies where the smallest adjacent MAE gap is under 2% of that "
      f"pair's mean: {(mae.min_gap_pct < 2).sum()} of {len(mae)}")
w = d.loc[d.swap_effect.idxmax()]
print(f"  worst case if ranks 2 and 3 are exchanged: {w.iso} {int(w.year)} "
      f"moves {w.swap_effect:.2f} pp (adjacent MAE gap {w.min_gap:.3f} points, "
      f"{w.min_gap_pct:.2f}% of mean)")
tight = mae.nsmallest(3, "min_gap")
print("  tightest separations: " + "; ".join(
    f"{i} {r.min_gap:.3f} pts ({r.min_gap_pct:.2f}%)" for i, r in tight.iterrows()))

big = d.reindex(d.diff_acc.abs().sort_values(ascending=False).index).head(3)
print("\n  largest divergences")
for _, r in big.iterrows():
    print(f"    {r.iso} {int(r.year)}: {r.owa:+.2f} -> {r.iowa_acc:+.2f} "
          f"(difference {r.diff_acc:.2f}), order {r.order_acc}")

sign = d[(np.sign(d.owa) != np.sign(d.iowa_acc)) & (d.diff_acc.abs() > 1e-9)]
print(f"\n  cells where the ordering flips the SIGN of the aggregate: {len(sign)}")
for _, r in sign.iterrows():
    print(f"    {r.iso} {int(r.year)}: {r.owa:+.2f} -> {r.iowa_acc:+.2f}")

if IN_COLAB:
    files.download("mae.csv")
    files.download("induced.csv")

Upload aggregates.csv


Saving aggregates.csv to aggregates.csv
aggregates.csv      aggregates.csv
Upload forecast_errors.csv


Saving forecast_errors.csv to forecast_errors.csv
forecast_errors.csv forecast_errors.csv

dropped for fewer than 3 usable rounds: HRV

36 economies with a complete record -> 72 cells

Section V-C
  IMF  most accurate in  16 | panel MAE 1.804
  OECD most accurate in  10 | panel MAE 1.911
  EC   most accurate in  10 | panel MAE 1.835

  accuracy ordering changes the aggregate in 63 of 72 cells
  mean |difference| 0.121 pp over all 72 cells  (0.138 over the 63 that change)
  Proposition 3 violations (difference > 0): 0

  recency ordering changes 67 of 72 cells, mean |difference| 0.151 pp over all cells (0.162 over those that change)
  (larger displacement is not evidence of a better inducing variable:
   the recency order is identical in every country)

  economies where the smallest adjacent MAE gap is under 2% of that pair's mean: 11 of 36
  worst case if ranks 2 and 3 are exchanged: IRL 2026 moves 0.55 pp (adjacent MAE gap 0.003 points, 0.05% of mean)
  tightest separations: CAN 0.00

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
"""
Supplementary figure — accuracy-induced minus magnitude-ordered aggregate,
by country-year. NOT included in the submitted paper; retained as
supplementary material for the repository.

Run as a follow-on CELL after the 03b_accuracy cell in the same notebook:
induced.csv is already on disk, so nothing is re-uploaded or recomputed.

Two changes from the earlier version:
  1. Ireland's bar was truncated at the axis limit and drawn exactly as long as
     Mexico's untruncated bar, so the eye could not tell they differ by 5x. It
     now ends in an arrow at the axis edge with its true value printed.
  2. legible label size, a validated colour pair, and hatching on 2027 so
     the year distinction survives grayscale printing. Single row of 36 --
     use \\begin{figure*} at full text width.

All differences are nonpositive by Proposition 3; the script checks this and
says so rather than leaving the reader to scan for a bar above zero.

WRITES  fig_induced_diff.pdf, fig_induced_diff.png
"""
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ALPHA = 0.8
YEARS = (2026, 2027)
K_OFFSCALE = 3.0        # truncate only bars deeper than K x the next deepest

C26, C27 = "#1F3B57", "#5C89B0"     # 12.0:1 and 4.0:1 on white
INK, GRID = "#222222", "#D5D5D5"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Nimbus Roman No9 L", "Times New Roman", "DejaVu Serif"],
    "font.size": 7, "pdf.fonttype": 42, "ps.fonttype": 42,
})

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQ = {"iso", "year", "diff_acc"}


def load():
    for name in ("induced.csv",) + tuple(sorted(os.listdir("."))):
        if not name.lower().endswith(".csv"):
            continue
        try:
            if REQ.issubset(pd.read_csv(name, nrows=1).columns):
                return pd.read_csv(name)
        except Exception:
            pass
    if IN_COLAB:
        print("Upload induced.csv")
        files.upload()
        return pd.read_csv("induced.csv")
    raise FileNotFoundError("induced.csv not found -- run the 03b_accuracy cell first")


d = load()
w = d.pivot(index="iso", columns="year", values="diff_acc")
w = w.reindex(w.min(axis=1).sort_values().index)          # deepest first
print(f"{len(w)} economies x {len(YEARS)} years = {len(d)} cells")

bad = (d.diff_acc > 1e-9).sum()
print(f"Proposition 3: {bad} violation(s) -- every difference must be <= 0")

# --- x-floor: clip only a bar that is genuinely off-scale, never one that is
#     merely the largest of a comparable set
depth = w.min(axis=1)
off = depth < K_OFFSCALE * depth.nsmallest(2).iloc[-1]
floor = float(depth[~off].min() * 1.10)
print(f"off-scale (deeper than {K_OFFSCALE}x the next): "
      f"{', '.join(w.index[off]) if off.any() else 'none'}")

fig, ax = plt.subplots(figsize=(7.16, 2.7))
ax.axhline(0, color=INK, lw=0.6, zorder=3)
for i, (iso, r) in enumerate(w.iterrows()):
    for yr, col, hatch, dx in ((YEARS[0], C26, "", -0.20),
                               (YEARS[1], C27, "///", +0.20)):
        v = r.get(yr, np.nan)
        if pd.isna(v):
            continue
        ax.bar(i + dx, max(v, floor), width=0.36, bottom=0, color=col,
               edgecolor="white", linewidth=0.35, hatch=hatch, zorder=2)
        if v < floor:                        # visibly truncated, not silently
            ax.plot(i + dx, floor, "v", color=col, ms=3.4, clip_on=False,
                    zorder=5)
            ax.annotate(f"{iso} {yr}: {v:.2f}", xy=(i + dx, floor),
                        xytext=(7, 3), textcoords="offset points",
                        fontsize=6.6, color=col, ha="left", va="bottom",
                        zorder=6, bbox=dict(fc="white", ec="none", pad=0.8))

ax.set_xticks(range(len(w)))
ax.set_xticklabels(w.index, rotation=90, fontsize=6.5)
ax.set_xlim(-0.7, len(w) - 0.3)
ax.set_ylim(floor, abs(floor) * 0.06)
ax.tick_params(labelsize=6.8, length=2, pad=1.5)
ax.set_ylabel(r"IOWAGDP$_{\rm acc}$ $-$ OWAGDP at $\alpha=%.1f$ (pp)" % ALPHA,
              fontsize=7.4)
ax.yaxis.grid(True, color=GRID, lw=0.4)
ax.set_axisbelow(True)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.spines["left"].set_linewidth(0.5)
ax.spines["bottom"].set_linewidth(0.5)

ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, fc=C26, ec="white", lw=0.35,
                                 label=str(YEARS[0])),
                   plt.Rectangle((0, 0), 1, 1, fc=C27, ec="white", lw=0.35,
                                 hatch="///", label=str(YEARS[1]))],
          frameon=False, loc="lower right", fontsize=7, handlelength=1.6,
          borderpad=0.2, labelspacing=0.3, ncol=2)

plt.tight_layout(pad=0.4)
for ext in ("pdf", "png"):
    plt.savefig(f"fig_induced_diff.{ext}", dpi=400, bbox_inches="tight")
plt.show()

nz = d.diff_acc.abs() > 1e-9
print(f"changed in {nz.sum()} of {len(d)} cells | mean |difference| "
      f"{d.diff_acc.abs().mean():.3f} pp over all cells")
print(f"deepest: " + "; ".join(
    f"{r.iso} {int(r.year)} {r.diff_acc:.2f}"
    for _, r in d.nsmallest(3, "diff_acc").iterrows()))

if IN_COLAB:
    files.download("fig_induced_diff.pdf")

36 economies x 2 years = 72 cells
Proposition 3: 0 violation(s) -- every difference must be <= 0
off-scale (deeper than 3.0x the next): IRL
changed in 63 of 72 cells | mean |difference| 0.121 pp over all cells
deepest: IRL 2026 -2.24; MEX 2026 -0.40; EST 2027 -0.35


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>